# 16 Timestamp Alignment Batch QC Template

Template for batch validation of timestamp alignment tables.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:

keys = (
    scan.Scan * session.Session * session.SessionUser * subject.User
    & f'initials = "{INITIALS}"'
    & f'session_datetime >= "{DATE_FROM}"'
).fetch("KEY")

summary_rows = []
for key in keys:
    te = trial.TrialEvent & key
    event_rows = event.Event & key

    trial_count = len(trial.Trial & key)
    trial_event_count = len(te)
    event_count = len(event_rows)
    camsync_count = len(behavior.CamSyncRecording & key)

    event_types = list((event_rows).fetch("event_type")) if event_count else []
    sync_like = sum(1 for e in event_types if "sync" in str(e).lower())

    summary_rows.append(
        {
            **key,
            "trial_count": trial_count,
            "trial_event_count": trial_event_count,
            "event_count": event_count,
            "camsync_count": camsync_count,
            "sync_like_events": sync_like,
        }
    )

summary = pd.DataFrame(summary_rows)
summary


In [ ]:

flags = summary[
    (summary["trial_count"] > 0)
    & (
        (summary["event_count"] == 0)
        | (summary["sync_like_events"] == 0)
        | (summary["camsync_count"] == 0)
    )
]

flags
